In [1]:
import numpy as np
import torch, torch.nn as nn

### 딥러닝 -> 이미지 분석 -> 합성곱(컨볼루션)
- Linear() : 퍼셉트론 여러개 연결한 레이어
- 컨볼루션 레이어 : 필터(분석하기 좋게 변환)

In [ ]:
# 이미지
img = np.array([[1,2,3,4,5],[6,7,8,9,10],[11,12,13,14,15],[16,17,18,19,20],[21,22,23,24,25]]) # 임의의 이미지
# 필터
kernel = np.array([[1,0,-1],[1,0,-1],[1,0,-1]]) # 수직 엣지 감지 

In [3]:
img.shape, kernel.shape

((5, 5), (3, 3))

In [5]:
out = np.zeros((3,3))


In [8]:
# 필터 (크기 3*3) 움직이면서 적용 
for i in range(3):
    for j in range(3):
        out[i,j] = (img[i:i+3,j:j+3] * kernel).sum()

print(out) # 특징맵(필터 거친 결과물)

[[-6. -6. -6.]
 [-6. -6. -6.]
 [-6. -6. -6.]]


In [10]:
x = torch.rand(1,1,28,28) #임의의 이미지 생성
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3) # 내부적으로 위의 필터 적용과 동일한 연산 수행 
out = conv(x)
print(out.shape)


torch.Size([1, 8, 26, 26])


### 풀링 / 스트라이드 / 패딩

In [13]:
# 풀링(압축)
# 맥스 풀링(제일 큰 특성만 남김), 에버리지풀링
x = torch.tensor([[[[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]]])
x.shape

torch.Size([1, 1, 4, 4])

In [14]:
# 스트라이드 : 이동하는 칸의 수 
pool = nn.MaxPool2d(kernel_size=2,stride=2) # 2*2로 계산 -> 첫 번째 계산(1,2,5,6)중 가장 큰 값 6 남음 
out = pool(x)
print(out.squeeze())

tensor([[ 6,  8],
        [14, 16]])


In [16]:
!uv add torchinfo

Resolved 76 packages in 316ms
Prepared 1 package in 32ms
Installed 1 package in 28ms
 + torchinfo==1.8.0


In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),  #커널 개수 16개
    nn.ReLU(),
    nn.MaxPool2d(2), # 커널 사이즈 2*2
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32*7*7, 10) # 원소의 개수, 출력 개수 
)

In [19]:
from torchinfo import summary

summary(model, input_size=[1,1,28,28])


Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [1, 10]                   --
├─Conv2d: 1-1                            [1, 16, 28, 28]           160
├─ReLU: 1-2                              [1, 16, 28, 28]           --
├─MaxPool2d: 1-3                         [1, 16, 14, 14]           --
├─Conv2d: 1-4                            [1, 32, 14, 14]           4,640
├─ReLU: 1-5                              [1, 32, 14, 14]           --
├─MaxPool2d: 1-6                         [1, 32, 7, 7]             --
├─Flatten: 1-7                           [1, 1568]                 --
├─Linear: 1-8                            [1, 10]                   15,690
Total params: 20,490
Trainable params: 20,490
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.05
Input size (MB): 0.00
Forward/backward pass size (MB): 0.15
Params size (MB): 0.08
Estimated Total Size (MB): 0.24

In [22]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32*7*7, 10)
)

summary(model, input_size=[1,1,28,28])

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [1, 10]                   --
├─Conv2d: 1-1                            [1, 16, 28, 28]           160
├─ReLU: 1-2                              [1, 16, 28, 28]           --
├─MaxPool2d: 1-3                         [1, 16, 14, 14]           --
├─Conv2d: 1-4                            [1, 32, 14, 14]           4,640
├─ReLU: 1-5                              [1, 32, 14, 14]           --
├─MaxPool2d: 1-6                         [1, 32, 7, 7]             --
├─Flatten: 1-7                           [1, 1568]                 --
├─Linear: 1-8                            [1, 10]                   15,690
Total params: 20,490
Trainable params: 20,490
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 1.05
Input size (MB): 0.00
Forward/backward pass size (MB): 0.15
Params size (MB): 0.08
Estimated Total Size (MB): 0.24

### 패딩

In [26]:
x = torch.rand(1,3,32,32)
conv1 = nn.Conv2d(3, 16, 3, stride=1, padding=1)
print(f'padding=1: {tuple(conv1(x).shape)}')
conv2= nn.Conv2d(3, 16, 3, stride=1, padding=0)
print(f'padding=0: {tuple(conv2(x).shape)}')
conv3 = nn.Conv2d(3, 16, 3, stride=2, padding=1)
print(f'stride=2, padding=1: {tuple(conv3(x).shape)}')

padding=1: (1, 16, 32, 32)
padding=0: (1, 16, 30, 30)
stride=2, padding=1: (1, 16, 16, 16)


### 합성곱 신경망(CNNs)
- 이미지, 음성 인식 분야 등 딥러닝 기술에 사용
- 컨볼루션 신경망( 컨볼루션 레이어,  풀링 레이어 )
- Lenet-5

LeNet

In [27]:
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, 5, padding=2)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.flatten(1) # 배치 차원 제외하고 평탄화
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

In [ ]:
m = LeNet()
tuple(m(torch.rand(1,2,28)).shape)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x784 and 400x120)

### 과적합 방지 기법
- BatchNorm
- dropout


In [33]:
block = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.BatchNorm2d(16),
    nn.ReLU()
)
x = torch.rand(4,1,28,28)
tuple(block(x).shape)

(4, 16, 28, 28)

In [ ]:
model = nn.Sequential(
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Dropout(p=0.5), # 학습이 잘 안되게 하는 역할(과적합 방지)
    nn.Linear(50, 10)
)
model.train()
x = torch.rand(2, 100)
model.eval()

Sequential(
  (0): Linear(in_features=100, out_features=50, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=50, out_features=10, bias=True)
)

Mnist

In [35]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.flatten(1) # 배치 차원 제외하고 평탄화
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

In [36]:
model = MnistCNN()
sum = 0
for p in model.parameters():
    print(p.numel())


288
32
18432
64
401408
128
1280
10


### MNist 데이터 적용해보기

In [37]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

trm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]) # 평균, 표준편차
train = datasets.MNIST(
    root = './data',
    train = True,
    download = True,
    transform = trm
)

loader = DataLoader(train, batch_size = 64, shuffle=True)

### 딥러닝을 구성하는 요소
- 퍼셉트론: y = wx + b
- MLP
- AN, AF(활성화함수 : 스텝, 시그모이드, 렐루, ...)
- 오차 (MSE, CE)
- 최적화 (경사하강법)
- 학습률 (너무 작으면 학습 시간이 오래 걸리거나 학습이 덜 되는 문제, 너무 크면 발산)
- 오차 업데이트 (역방향 계산 : 체인룰)
- 학습 횟수 
- 배치(4차원), 이미지(3차원), 매트릭스(2차원), 벡터(1차원), 스칼라(0차원)
- 전처리 (정규화, 표준화)

### 합성곱 신경망 (CNN)
- kernel(필터): 갯수(필터 몇 개를 적용할 것인가)
- 커널 사이즈 : 3*3 (너무 크지 않게, 이미지에 따라 사이즈 조절)
- stride: 몇 칸씩 이동하면서(건너뛰면서) 계산
- pooling: Max, average -> 주로 Maxpooling
- 활성화 함수 : ReLU(이미지에 최적화 -> 색은 0~255 사이의 값 -> ReLU는 0보다 작은 값(결측치)은 0으로 치환하기 때문(0보다 큰 값은 원래값 유지))


In [7]:
import torch
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.init

torch.manual_seed(9)

In [10]:
# 학습률
learning_rate = 0.001
# 학습횟수
training_epochs = 15
# 배치 사이즈
batch_size = 100
# mnist 데이터셋 
mnist_train = datasets.MNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
mnist_test = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor(), download=True)

# 배치 사이즈로 데이터 준비
data_loader = torch.utils.data.DataLoader(
    dataset=mnist_train, batch_size=batch_size, shuffle=True, drop_last=True)




In [16]:
# 모델 정의
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # 첫 번째 레이어(100,1,28,28) -> (100,32,14,14)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1,32,kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # 두 번째 레이어(100,32,14,14) -> (100,32,7,7)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32,64,kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # MLP 완전 연결 레이어(100,64,7,7) -> (100,10)
        self.fc = torch.nn.Linear(64*7*7, 10, bias=True)

        # 가중치(w) 초기화
        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1) # flatten, reshape과 동일
        out = self.fc(out)
        return out
    

model = CNN()

In [ ]:
# 손실함수(오차 계산) -> mnist는 분류하는 문제이므로 CrossEntropyLoss 사용
criterion = torch.nn.CrossEntropyLoss()
# 경사하강법 알고리즘 -> Adam 사용
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
# 전체 배치 수 -> 1 epoch마다 전체 데이터를 몇 번 반복하는지 계산
total_batch = len(data_loader)

In [19]:
# 학습 
for epoch in range(training_epochs):

    avg_cost = 0 # epoch당 평균 비용(손실) 저장 
    for X, y in data_loader: # 배치 사이즈로 데이터 준비 -> 100개씩 묶어서 X, y에 저장
       
        optimizer.zero_grad() # 기울기 초기화
        hypothesis = model(X) # 모델을 통해 예측값 계산 -> 순전파 계산
        cost = criterion(hypothesis, y) # 예측값과 실제값 간의 손실(오차) 계산
        cost.backward() # 역전파 계산
        optimizer.step() # 가중치 업데이트
        avg_cost += cost / total_batch # 평균 비용 계산
    print(f'Epoch: {epoch+1}, cost: {avg_cost:.4f}')
    print('Epoch: {:>4}, cost: {:>9}'.format(epoch+1, avg_cost))

Epoch: 1, cost: 2.3131
Epoch:    1, cost: 2.313060998916626
Epoch: 2, cost: 2.3131
Epoch:    2, cost: 2.3130598068237305
Epoch: 3, cost: 2.3131
Epoch:    3, cost: 2.313061237335205
Epoch: 4, cost: 2.3131
Epoch:    4, cost: 2.313060998916626
Epoch: 5, cost: 2.3131
Epoch:    5, cost: 2.313060998916626
Epoch: 6, cost: 2.3131
Epoch:    6, cost: 2.3130595684051514
Epoch: 7, cost: 2.3131
Epoch:    7, cost: 2.3130605220794678
Epoch: 8, cost: 2.3131
Epoch:    8, cost: 2.313060760498047
Epoch: 9, cost: 2.3131
Epoch:    9, cost: 2.313060998916626
Epoch: 10, cost: 2.3131
Epoch:   10, cost: 2.3130602836608887
Epoch: 11, cost: 2.3131
Epoch:   11, cost: 2.313060760498047
Epoch: 12, cost: 2.3131
Epoch:   12, cost: 2.3130602836608887
Epoch: 13, cost: 2.3131
Epoch:   13, cost: 2.3130600452423096
Epoch: 14, cost: 2.3131
Epoch:   14, cost: 2.3130605220794678
Epoch: 15, cost: 2.3131
Epoch:   15, cost: 2.3130593299865723


In [21]:
# 테스트
with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float()
    y_test = mnist_test.test_labels
    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction, 1) == y_test
    accuracy = correct_prediction.float().mean()
# print(f'Test Accuracy: {accuracy:.4f}')
    print('Accuracy:', accuracy.item())


c:\Users\UserK\work\06_pytorch_basic\.venv\Lib\site-packages\torchvision\datasets\mnist.py:81: UserWarning: test_data has been renamed data
  warnings.warn("test_data has been renamed data")
c:\Users\UserK\work\06_pytorch_basic\.venv\Lib\site-packages\torchvision\datasets\mnist.py:71: UserWarning: test_labels has been renamed targets
  warnings.warn("test_labels has been renamed targets")


Accuracy: 0.09700000286102295
